# Model 1: Custom CNN Baseline
### Retinal OCT Disease Classification (Kermany 2018 Dataset)

**Classes:** CNV | DME | DRUSEN | NORMAL  
**Challenges Addressed:** Class Imbalance, Speckle Noise, Overfitting, Inter-Class Similarity

In [ ]:
# ─────────────────────────────────────────────
# SECTION 1: IMPORTS
# ─────────────────────────────────────────────
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (
    classification_report, confusion_matrix,
    cohen_kappa_score, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)
print('GPU Available:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ─────────────────────────────────────────────
# SECTION 2: CONFIGURATION
# ─────────────────────────────────────────────
# UPDATE THIS PATH to your Kermany dataset root
DATASET_PATH = './OCT2017'  # Root folder containing train/ test/ val/ subfolders

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 30
SEED        = 42
NUM_CLASSES = 4
CLASS_NAMES = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR   = os.path.join(DATASET_PATH, 'val')
TEST_DIR  = os.path.join(DATASET_PATH, 'test')

tf.random.set_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 3: CLASS IMBALANCE — COMPUTE WEIGHTS
# Challenge: CNV ~37k vs DRUSEN ~8k
# Fix: Weighted loss — penalise minority class errors more
# ─────────────────────────────────────────────
def get_class_weights(train_dir, class_names):
    counts = [len(os.listdir(os.path.join(train_dir, c))) for c in class_names]
    total  = sum(counts)
    weights = {i: total / (NUM_CLASSES * c) for i, c in enumerate(counts)}
    print('\nClass counts:', dict(zip(class_names, counts)))
    print('Class weights:', weights)
    return weights

class_weights = get_class_weights(TRAIN_DIR, CLASS_NAMES)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 4: DATA GENERATORS
# Challenge: Noisy images → Gaussian noise + aggressive augmentation
# Challenge: Overfitting  → Heavy augmentation on training only
# ─────────────────────────────────────────────
def add_gaussian_noise(image):
    """Simulate OCT speckle noise removal via slight noise injection for robustness."""
    noise = np.random.normal(0, 0.01, image.shape)
    return np.clip(image + noise, 0, 1)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    preprocessing_function=add_gaussian_noise  # Noise robustness
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED, shuffle=True
)
val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED, shuffle=False
)
test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', seed=SEED, shuffle=False
)

print('\nClass indices:', train_gen.class_indices)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 5: VISUALISE SAMPLE IMAGES
# ─────────────────────────────────────────────
def visualise_samples(generator, class_names, n=8):
    images, labels = next(generator)
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i], cmap='gray')
        ax.set_title(class_names[np.argmax(labels[i])], fontsize=11)
        ax.axis('off')
    plt.suptitle('Sample OCT Images (Augmented)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('cnn_sample_images.png', dpi=150)
    plt.show()

visualise_samples(train_gen, CLASS_NAMES)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 6: BUILD CUSTOM CNN
# Challenge: Overfitting   → BatchNorm + Dropout + L2 regularisation
# Challenge: Inter-class similarity → Deeper conv blocks + GlobalAvgPool
# ─────────────────────────────────────────────
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=4):
    model = models.Sequential(name='Custom_CNN')

    # Block 1
    model.add(layers.Conv2D(32, (3,3), padding='same', activation='relu',
                             kernel_regularizer=regularizers.l2(1e-4),
                             input_shape=input_shape))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(32, (3,3), padding='same', activation='relu',
                             kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2,2))
    model.add(layers.Dropout(0.25))

    # Block 2
    model.add(layers.Conv2D(64, (3,3), padding='same', activation='relu',
                             kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(64, (3,3), padding='same', activation='relu',
                             kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2,2))
    model.add(layers.Dropout(0.25))

    # Block 3
    model.add(layers.Conv2D(128, (3,3), padding='same', activation='relu',
                             kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(128, (3,3), padding='same', activation='relu',
                             kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2,2))
    model.add(layers.Dropout(0.3))

    # Block 4
    model.add(layers.Conv2D(256, (3,3), padding='same', activation='relu',
                             kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D(2,2))
    model.add(layers.Dropout(0.3))

    # Classifier Head
    model.add(layers.GlobalAveragePooling2D())  # Better than Flatten for generalisation
    model.add(layers.Dense(256, activation='relu',
                            kernel_regularizer=regularizers.l2(1e-4)))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

cnn_model = build_custom_cnn()
cnn_model.summary()

In [ ]:
# ─────────────────────────────────────────────
# SECTION 7: COMPILE
# ─────────────────────────────────────────────
cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 8: CALLBACKS
# ─────────────────────────────────────────────
callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_cnn.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

In [ ]:
# ─────────────────────────────────────────────
# SECTION 9: TRAIN (with class_weights for imbalance)
# ─────────────────────────────────────────────
history = cnn_model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    class_weight=class_weights,  # KEY: handles class imbalance
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 10: TRAINING CURVES
# ─────────────────────────────────────────────
def plot_history(history, model_name='Custom CNN'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'],   label='Train Acc')
    axes[0].plot(history.history['val_accuracy'], label='Val Acc')
    axes[0].set_title(f'{model_name} — Accuracy')
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(history.history['loss'],      label='Train Loss')
    axes[1].plot(history.history['val_loss'],  label='Val Loss')
    axes[1].set_title(f'{model_name} — Loss')
    axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(f'{model_name.replace(" ","_")}_curves.png', dpi=150)
    plt.show()

plot_history(history)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 11: EVALUATION HELPER FUNCTIONS
# ─────────────────────────────────────────────
def evaluate_model(model, test_gen, class_names, model_name='Model'):
    test_gen.reset()
    y_pred_prob = model.predict(test_gen, verbose=1)
    y_pred      = np.argmax(y_pred_prob, axis=1)
    y_true      = test_gen.classes

    # ── Accuracy
    acc = np.mean(y_pred == y_true)
    print(f'\n{model_name} — Test Accuracy: {acc*100:.2f}%')

    # ── Classification Report (Precision, Recall, F1)
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=class_names))

    # ── Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} — Confusion Matrix', fontsize=13, fontweight='bold')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{model_name.replace(" ","_")}_confusion_matrix.png', dpi=150)
    plt.show()

    # ── Cohen's Kappa
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"Cohen's Kappa: {kappa:.4f}")

    # ── AUC-ROC (One-vs-Rest)
    y_true_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    auc_scores = {}
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
        auc = roc_auc_score(y_true_bin[:, i], y_pred_prob[:, i])
        auc_scores[cls] = auc
        plt.plot(fpr, tpr, label=f'{cls} (AUC={auc:.3f})')
    plt.plot([0,1],[0,1],'k--')
    plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.title(f'{model_name} — AUC-ROC Curve', fontsize=13, fontweight='bold')
    plt.legend(); plt.grid(True)
    plt.tight_layout()
    plt.savefig(f'{model_name.replace(" ","_")}_roc.png', dpi=150)
    plt.show()
    print('AUC per class:', auc_scores)

    return {'accuracy': acc, 'kappa': kappa, 'auc': auc_scores,
            'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_pred_prob}

cnn_results = evaluate_model(cnn_model, test_gen, CLASS_NAMES, 'Custom CNN')

In [ ]:
# ─────────────────────────────────────────────
# SECTION 12: GRAD-CAM EXPLAINABILITY
# Challenge: Inter-class similarity → show WHAT the model is looking at
# ─────────────────────────────────────────────
import cv2

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_index.numpy()

def display_gradcam(model, test_gen, class_names, last_conv='conv2d_7'):
    test_gen.reset()
    images, labels = next(test_gen)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for i, ax in enumerate(axes.flat):
        img   = images[i]
        input_img = np.expand_dims(img, axis=0)
        heatmap, pred_idx = make_gradcam_heatmap(input_img, model, last_conv)
        heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
        heatmap_rgb     = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
        heatmap_rgb     = cv2.cvtColor(heatmap_rgb, cv2.COLOR_BGR2RGB) / 255.0
        overlay = 0.6 * img + 0.4 * heatmap_rgb
        ax.imshow(np.clip(overlay, 0, 1))
        true_lbl = class_names[np.argmax(labels[i])]
        pred_lbl = class_names[pred_idx]
        color = 'green' if true_lbl == pred_lbl else 'red'
        ax.set_title(f'True:{true_lbl}\nPred:{pred_lbl}', color=color, fontsize=9)
        ax.axis('off')
    plt.suptitle('Grad-CAM Heatmaps — Custom CNN', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('cnn_gradcam.png', dpi=150)
    plt.show()

# Find last conv layer name from model summary
last_conv_layer = [l.name for l in cnn_model.layers if 'conv2d' in l.name][-1]
print('Last conv layer:', last_conv_layer)
display_gradcam(cnn_model, test_gen, CLASS_NAMES, last_conv=last_conv_layer)

In [ ]:
# ─────────────────────────────────────────────
# SECTION 13: SAVE RESULTS
# ─────────────────────────────────────────────
cnn_model.save('custom_cnn_final.keras')
print('Model saved.')
print(f"\n=== SUMMARY ===")
print(f"Test Accuracy : {cnn_results['accuracy']*100:.2f}%")
print(f"Cohen's Kappa : {cnn_results['kappa']:.4f}")
for cls, auc in cnn_results['auc'].items():
    print(f"AUC [{cls}]     : {auc:.4f}")